In [17]:
# O notebook roda dentro de notebooks/; adicionamos a raiz do projeto ao
# caminho de importação para conseguir fazer "from src...."
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "vscode"
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from src.ml.features import FEATURES_CLUSTER, features_clusterizacao

In [18]:
feats = features_clusterizacao().dropna(subset=FEATURES_CLUSTER)
print(len(feats), "municípios")
feats.head()

141 municípios


,municipio,part_soja,part_milho,part_algodao,part_outras,area_media_ha,indice_produtividade,crescimento_area_log,log_area_media
cod_municipio,,,,,,,,,
5100102,Acorizal,0.000000,0.000000,0.000000,1.000000,10.2,0.260743,-2.614960,2.415914
5100201,Água Boa,0.768874,0.209573,0.006023,0.015530,265300.8,0.943824,0.736492,12.488623
5100250,Alta Floresta,0.628616,0.350295,0.000000,0.021089,109060.8,1.003078,2.483562,11.599670
5100300,Alto Araguaia,0.693197,0.195218,0.025439,0.086146,65567.8,1.000419,0.322803,11.090855
5100359,Alto Boa Vista,0.594437,0.399789,0.000000,0.005773,55426.2,0.994592,1.250642,10.922826


In [19]:
px.histogram(feats, x="area_media_ha", nbins=40, title="Área média (ha): assimétrica").show()
px.histogram(feats, x="log_area_media", nbins=40, title="log(área): bem mais equilibrada").show()

In [20]:
X = StandardScaler().fit_transform(feats[FEATURES_CLUSTER])
# Conferência: médias ~0 e desvios ~1 em todas as colunas
pd.DataFrame(X, columns=FEATURES_CLUSTER).describe().loc[["mean", "std"]].round(2)

,part_soja,part_milho,part_algodao,part_outras,log_area_media,indice_produtividade,crescimento_area_log
mean,0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [21]:
resultados = []
for k in range(2, 11):
    # random_state fixo: o resultado é o mesmo a cada execução (reprodutível)
    # n_init=10: roda 10 vezes com centros iniciais diferentes e fica com a melhor
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    resultados.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silhueta": silhouette_score(X, modelo.labels_),
    })

resultados = pd.DataFrame(resultados)
px.line(resultados, x="k", y="inercia", markers=True, title="Método do cotovelo").show()
px.line(resultados, x="k", y="silhueta", markers=True, title="Silhueta média").show()
resultados.round(3)

,k,inercia,silhueta
0,2,688.779,0.570
1,3,532.843,0.576
2,4,401.267,0.299
3,5,329.964,0.300
4,6,269.073,0.315
5,7,222.378,0.329
6,8,200.977,0.323
7,9,183.117,0.278
8,10,166.833,0.282


In [22]:
for k in [2, 3, 4, 5, 6, 7]:
    rotulos = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    tamanhos = pd.Series(rotulos).value_counts().sort_index().tolist()
    print(f"k={k}: municípios por grupo = {tamanhos}")

k=2: municípios por grupo = [126, 15]
k=3: municípios por grupo = [124, 10, 7]
k=4: municípios por grupo = [55, 10, 7, 69]
k=5: municípios por grupo = [44, 7, 17, 8, 65]
k=6: municípios por grupo = [1, 43, 7, 13, 9, 68]
k=7: municípios por grupo = [13, 31, 9, 58, 1, 22, 7]


In [23]:
import numpy as np

def perfil(k, dados=feats, matriz=X):
    """Resume cada grupo nas unidades originais, para podermos interpretar."""
    rotulos = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(matriz)
    tabela = dados.assign(cluster=rotulos)

    resumo = tabela.groupby("cluster").agg(
        municipios=("municipio", "count"),
        part_soja=("part_soja", "mean"),
        part_milho=("part_milho", "mean"),
        part_algodao=("part_algodao", "mean"),
        part_outras=("part_outras", "mean"),
        area_mediana_ha=("area_media_ha", "median"),  # mediana: resiste a extremos
        indice_produtividade=("indice_produtividade", "mean"),
        crescimento_log=("crescimento_area_log", "median"),
    )
    # De log para "vezes": exp(0,7) ≈ 2,0 -> a área dobrou
    resumo["crescimento_vezes"] = np.exp(resumo.pop("crescimento_log"))

    # Exemplos: os 3 maiores municípios de cada grupo
    exemplos = (
        tabela.sort_values("area_media_ha", ascending=False)
        .groupby("cluster")["municipio"]
        .apply(lambda nomes: ", ".join(nomes.head(3)))
    )
    return resumo.join(exemplos.rename("exemplos")).round(2)

perfil(3)

,municipios,part_soja,part_milho,part_algodao,part_outras,area_mediana_ha,indice_produtividade,crescimento_vezes,exemplos
cluster,,,,,,,,,
0,124,0.63,0.30,0.03,0.03,79450.6,0.98,2.32,"Sorriso, Campo Novo do Parecis, Diamantino"
1,10,0.11,0.64,0.00,0.24,219.7,0.68,1.34,"Colniza, Araputanga, Porto Estrela"
2,7,0.06,0.03,0.00,0.91,12270.0,1.01,1.09,"Barra do Bugres, Denise, Nova Olímpia"


In [24]:
perfil(4)

,municipios,part_soja,part_milho,part_algodao,part_outras,area_mediana_ha,indice_produtividade,crescimento_vezes,exemplos
cluster,,,,,,,,,
0,55,0.58,0.32,0.07,0.03,240638.0,1.00,1.51,"Sorriso, Campo Novo do Parecis, Diamantino"
1,10,0.11,0.64,0.00,0.24,219.7,0.68,1.34,"Colniza, Araputanga, Porto Estrela"
2,7,0.06,0.03,0.00,0.91,12270.0,1.01,1.09,"Barra do Bugres, Denise, Nova Olímpia"
3,69,0.68,0.29,0.00,0.03,33255.0,0.97,4.02,"Água Boa, São José do Xingu, Marcelândia"


In [25]:
perfil(5)

,municipios,part_soja,part_milho,part_algodao,part_outras,area_mediana_ha,indice_produtividade,crescimento_vezes,exemplos
cluster,,,,,,,,,
0,44,0.68,0.29,0.00,0.04,14680.7,0.97,7.42,"Juara, Novo Mundo, Alta Floresta"
1,7,0.06,0.03,0.00,0.91,12270.0,1.01,1.09,"Barra do Bugres, Denise, Nova Olímpia"
2,17,0.54,0.29,0.15,0.02,248742.2,1.00,1.27,"Campo Novo do Parecis, Diamantino, Sapezal"
3,8,0.08,0.62,0.00,0.30,116.5,0.62,1.18,"Colniza, Porto Estrela, Rio Branco"
4,65,0.62,0.33,0.02,0.03,118114.0,0.99,1.71,"Sorriso, Nova Mutum, Nova Ubiratã"


In [26]:
perfil(7)

,municipios,part_soja,part_milho,part_algodao,part_outras,area_mediana_ha,indice_produtividade,crescimento_vezes,exemplos
cluster,,,,,,,,,
0,13,0.54,0.28,0.17,0.02,248742.2,1.00,1.25,"Campo Novo do Parecis, Sapezal, Lucas do Rio V..."
1,31,0.77,0.20,0.00,0.04,13202.8,0.96,3.45,"Vila Bela da Santíssima Trindade, Nova Xavanti..."
2,9,0.13,0.71,0.00,0.16,285.6,0.72,1.38,"Colniza, Araputanga, Porto Estrela"
3,58,0.60,0.34,0.03,0.03,174629.7,0.99,1.71,"Sorriso, Diamantino, Nova Mutum"
4,1,0.00,0.00,0.00,1.00,10.2,0.26,0.07,Acorizal
5,22,0.59,0.36,0.00,0.04,30965.3,0.98,8.91,"Juara, Novo Mundo, Alta Floresta"
6,7,0.06,0.03,0.00,0.91,12270.0,1.01,1.09,"Barra do Bugres, Denise, Nova Olímpia"


In [27]:
rotulos = KMeans(n_clusters=6, random_state=42, n_init=10).fit_predict(X)
tabela = feats.assign(cluster=rotulos)
contagem = tabela["cluster"].value_counts()
isolado = tabela[tabela["cluster"].isin(contagem[contagem == 1].index)]
isolado.drop(columns="cluster").round(3).T

cod_municipio,5100102
municipio,Acorizal
part_soja,0.0
part_milho,0.0
part_algodao,0.0
part_outras,1.0
area_media_ha,10.2
indice_produtividade,0.261
crescimento_area_log,-2.615
log_area_media,2.416


In [28]:
for limite in [100, 500, 1_000, 2_000, 5_000]:
    abaixo = feats[feats["area_media_ha"] < limite]
    print(f"< {limite:>5} ha: {len(abaixo):>2} municípios -> {', '.join(abaixo['municipio'])}")

<   100 ha:  4 municípios -> Acorizal, Barão de Melgaço, Reserva do Cabaçal, Várzea Grande
<   500 ha:  7 municípios -> Acorizal, Barão de Melgaço, Jauru, Ponte Branca, Reserva do Cabaçal, Rio Branco, Várzea Grande
<  1000 ha: 11 municípios -> Acorizal, Barão de Melgaço, Figueirópolis D'Oeste, Indiavaí, Jauru, Ponte Branca, Reserva do Cabaçal, Rio Branco, São José do Povo, Rondolândia, Várzea Grande
<  2000 ha: 14 municípios -> Acorizal, Barão de Melgaço, Figueirópolis D'Oeste, Indiavaí, Jauru, Ponte Branca, Porto Estrela, São José dos Quatro Marcos, Reserva do Cabaçal, Rio Branco, São José do Povo, Rondolândia, Vale de São Domingos, Várzea Grande
<  5000 ha: 28 municípios -> Acorizal, Araguainha, Araputanga, Barão de Melgaço, Castanheira, Colniza, Cuiabá, Curvelândia, Figueirópolis D'Oeste, Glória D'Oeste, Indiavaí, Jangada, Jauru, Juruena, Novo Santo Antônio, Pontal do Araguaia, Ponte Branca, Porto Esperidião, Porto Estrela, São José dos Quatro Marcos, Reserva do Cabaçal, Rio Branco,

In [29]:
AREA_MINIMA_HA = 500

relevantes = feats[feats["area_media_ha"] >= AREA_MINIMA_HA]
print(f"{len(relevantes)} no K-Means | {len(feats) - len(relevantes)} no perfil de regra")

# O escalonamento é refeito SÓ com os relevantes: média e desvio sem os extremos
X_rel = StandardScaler().fit_transform(relevantes[FEATURES_CLUSTER])

linhas = []
for k in range(2, 9):
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_rel)
    linhas.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silhueta": silhouette_score(X_rel, modelo.labels_),
        "tamanhos": sorted(pd.Series(modelo.labels_).value_counts().tolist(), reverse=True),
    })
pd.DataFrame(linhas).round(3)

134 no K-Means | 7 no perfil de regra


,k,inercia,silhueta,tamanhos
0,2,696.512,0.561,"[126, 8]"
1,3,530.321,0.260,"[68, 58, 8]"
2,4,411.293,0.288,"[63, 58, 7, 6]"
3,5,338.529,0.287,"[65, 40, 16, 7, 6]"
4,6,288.713,0.285,"[55, 30, 20, 16, 7, 6]"
5,7,255.045,0.298,"[54, 33, 19, 13, 7, 6, 2]"
6,8,228.400,0.246,"[35, 31, 26, 16, 13, 7, 4, 2]"


In [30]:
perfil(4, relevantes, X_rel)

,municipios,part_soja,part_milho,part_algodao,part_outras,area_mediana_ha,indice_produtividade,crescimento_vezes,exemplos
cluster,,,,,,,,,
0,58,0.58,0.33,0.06,0.02,223564.0,1.00,1.55,"Sorriso, Campo Novo do Parecis, Diamantino"
1,7,0.06,0.03,0.00,0.91,12270.0,1.01,1.09,"Barra do Bugres, Denise, Nova Olímpia"
2,6,0.24,0.65,0.00,0.12,3824.7,0.84,3.57,"Cotriguaçu, Juruena, Colniza"
3,63,0.69,0.27,0.00,0.03,31472.8,0.97,4.02,"Água Boa, Itanhangá, Juara"


In [31]:
perfil(5, relevantes, X_rel)

,municipios,part_soja,part_milho,part_algodao,part_outras,area_mediana_ha,indice_produtividade,crescimento_vezes,exemplos
cluster,,,,,,,,,
0,65,0.60,0.35,0.02,0.03,132958.4,0.99,1.89,"Sorriso, Nova Mutum, Nova Ubiratã"
1,7,0.06,0.03,0.00,0.91,12270.0,1.01,1.09,"Barra do Bugres, Denise, Nova Olímpia"
2,16,0.54,0.29,0.15,0.02,318114.9,1.00,1.26,"Campo Novo do Parecis, Diamantino, Sapezal"
3,40,0.73,0.23,0.00,0.04,13761.3,0.96,4.93,"Vila Bela da Santíssima Trindade, Nova Xavanti..."
4,6,0.24,0.65,0.00,0.12,3824.7,0.84,3.57,"Cotriguaçu, Juruena, Colniza"
